In [69]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json
import os
import sqlite3
import uuid

In [70]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
DB = "tickets.db"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [71]:
system_message = """
You are an AI support assistant for a hosting provider.

You have no technical knowledge of your own. Never answer a technical question
from your own knowledge, never guess, and never speculate about causes.
Every technical answer must come from the ask_specialist tool.
Do not tell the customer that you lack knowledge, and never mention your tools.
Keep every reply to at most two sentences.

Follow this workflow every time:

1. Listen to the customer's problem.
2. If the customer mentions a ticket ID, call get_ticket first and use what it
   returns as context. Do not stop here - continue with the workflow.
3. Decide which specialist the problem needs: network, billing or security.
   If you cannot tell, ask the customer one clarifying question first.
4. Call ask_specialist. The specialist remembers your prior messages to them,
   and the tool returns the full conversation as JSON.
   FIRST call: include the full problem and all relevant facts.
   FOLLOW-UP calls: send ONLY the new fact in one or two sentences — do not
   repeat anything already in the returned history.
5. If the specialist's answer is something the customer can act on themselves,
   relay it and confirm the problem is solved.
6. If resolving it requires access or authority the customer does not have,
   call create_ticket and put the specialist's reasoning into specialist_notes.
   Never invent a ticket number - use the number the tool returns. Once a ticket
   exists for a problem, refer to it by that number and never create a second one.

Human intervention IS needed when someone with access to our infrastructure has
to act:
- Physical work in the datacenter: a failed disk, a RAM or GPU upgrade, a move
  to a different rack.
- Server-side actions the customer cannot perform: restoring a database from a
  snapshot, pulling raw server logs, releasing a blocked IP.
- Decisions that cost money or need authorization: refunding a duplicate charge,
  SLA credit, an early plan change.
- A confirmed security incident: a compromised account, outbound spam from the
  customer's server, anything needing forensics or an account lock.
- The specialist says they need data the customer cannot provide.

Human intervention is NOT needed when the specialist's answer is enough on its
own:
- The customer can carry out the fix in their control panel or their own code:
  pointing a DNS record at a new IP, raising a PHP memory limit, fixing a
  misconfigured cron.
- The question is about how something works or what the plan includes: backup
  retention, bandwidth limits, what a warning in the panel means.
- The specialist identifies the cause and the fix is on the customer's side:
  a slow site caused by an unindexed query, a certificate that failed to renew
  because the old validation record is still in DNS.
- The answer is reassurance or a check: confirming an email is phishing,
  confirming a maintenance window is already scheduled.
"""


In [72]:
SPECIALIST_PREAMBLE = """You work for a hosting provider. You are answering a
support assistant, not the customer - never greet anyone, never address the
customer directly, and never ask a follow-up question. You may receive follow-up
messages in the same thread; use the conversation history above and respond only
to what is new in the latest message. Do not restate facts already established
in earlier turns.

If the question is missing something you need, do not guess a cause. Say what is
missing instead.

Answer in exactly this format:

ASSESSMENT: at most three sentences, technical, no filler. Address only the new
information in the latest message — do not summarize the whole case from scratch.
VERDICT: one of CUSTOMER_CAN_FIX, NEEDS_HUMAN, NEED_MORE_INFO
NEXT STEP: one concrete action - what the customer should do, what our engineer
should do, or what the assistant should ask the customer.

Use NEEDS_HUMAN when resolving this requires access or authority the customer
does not have. Use CUSTOMER_CAN_FIX when your NEXT STEP is something the
customer can carry out themselves.
"""

SPECIALISTS = {
    "network": SPECIALIST_PREAMBLE + """
You are a senior network engineer. Your scope is DNS, routing, IP allocation,
firewall rules, TLS certificates, CDN behaviour and connectivity problems.
Name the specific record, port, protocol or config directive involved. If the
symptom could come from either our network or the customer's own application,
say which one you think it is and what would distinguish them.
""",

    "billing": SPECIALIST_PREAMBLE + """
You are a senior billing specialist. Your scope is invoices, charges, plan
limits, upgrades, downgrades and renewals. State exactly what a plan does and
does not include. Never promise a refund, a credit or a discount - if the
customer is owed money, that is NEEDS_HUMAN and someone with authority decides.
Never quote a price you were not given.
""",

    "security": SPECIALIST_PREAMBLE + """
You are a security specialist. Your scope is compromised accounts, malware,
outbound spam, phishing, brute-force attempts and suspicious logins. Be
conservative: when the evidence is ambiguous, treat it as a possible incident
rather than as noise. Anything that looks like an active compromise is
NEEDS_HUMAN - containment, log preservation and forensics are never left to the
customer. Never instruct the customer to delete anything before logs are pulled.
""",
}

In [73]:
network_specialist_history = []
billing_specialist_history = []
security_specialist_history = []


def ask_specialist(domain, message):
    print(f"Specialist called for domain {domain}")
    system = SPECIALISTS[domain]

    match domain:
        case "network":
            history = network_specialist_history
        case "billing":
            history = billing_specialist_history
        case "security":
            history = security_specialist_history
        case _:
            raise ValueError(f"Unknown domain: {domain}")

    user_message = {"role": "user", "content": message}
    messages = [{"role": "system", "content": system}] + history + [user_message]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    reply = response.choices[0].message.content
    history.append(user_message)
    history.append({"role": "assistant", "content": reply})

    return history


In [74]:
def init_db():
    with sqlite3.connect(DB) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS tickets (
                ticket_id TEXT PRIMARY KEY,
                domain TEXT NOT NULL,
                status TEXT NOT NULL,
                specialist_notes TEXT NOT NULL
            )
        """)


init_db()


def create_ticket(domain, status, specialist_notes):
    print(f'Creating ticket for domain {domain}')

    ticket_id = str(uuid.uuid4())[:8]

    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO tickets (ticket_id, domain, status, specialist_notes) VALUES (?, ?, ?, ?)',
            (ticket_id, domain, status, specialist_notes),
        )

    return f"Ticket {ticket_id} created with status {status} for {domain} specialist."


def get_ticket(ticket_id):
    print(f'Getting ticket for ticket_id {ticket_id}')

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT ticket_id, domain, status, specialist_notes FROM tickets WHERE ticket_id = ?', (ticket_id,))
        ticket = cursor.fetchone()

    if not ticket:
        return f"No ticket found with ID {ticket_id}."

    ticket_id, domain, status, specialist_notes = ticket
    return f"Ticket {ticket_id}: domain={domain}, status={status}, notes={specialist_notes}"



In [75]:
get_ticket_function = {
    "name": "get_ticket",
    "description": "Look up an existing support ticket by its ID.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticket_id": {
                "type": "string",
                "description": "The ID of the ticket to retrieve.",
            }
        },
        "required": ["ticket_id"],
        "additionalProperties": False,
    },
}

create_ticket_function = {
    "name": "create_ticket",
    "description": "Create a support ticket when human intervention is needed.",
    "parameters": {
        "type": "object",
        "properties": {
            "domain": {
                "type": "string",
                "enum": ["network", "billing", "security"],
                "description": "Which specialist domain this ticket belongs to.",
            },
            "status": {
                "type": "string",
                "description": "The status of the ticket, e.g. open.",
            },
            "specialist_notes": {
                "type": "string",
                "description": "The specialist's reasoning and recommended actions.",
            },
        },
        "required": ["domain", "status", "specialist_notes"],
        "additionalProperties": False,
    },
}

ask_specialist_function = {
    "name": "ask_specialist",
    "description": (
        "Ask a specialist a technical question. The specialist remembers prior messages "
        "in this thread. FIRST call: include the full problem. FOLLOW-UP calls: send ONLY "
        "new facts — do not repeat the history. "
        "Returns a JSON array: "
        '[{"role": "user", "content": "..."}, {"role": "assistant", "content": "ASSESSMENT: ...\\nVERDICT: ...\\nNEXT STEP: ..."}, ...]'
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "domain": {
                "type": "string",
                "enum": ["network", "billing", "security"],
                "description": "Which specialist to consult.",
            },
            "message": {
                "type": "string",
                "description": (
                    "First call: the full question plus all relevant context. "
                    "Follow-up call: only what changed or what the customer just answered."
                ),
            },
        },
        "required": ["domain", "message"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": get_ticket_function},
    {"type": "function", "function": ask_specialist_function},
    {"type": "function", "function": create_ticket_function},
]

In [76]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_ticket":
            ticket_id = arguments.get("ticket_id")
            response = get_ticket(ticket_id)

            responses.append({
                "role": "tool",
                "content": response,
                "tool_call_id": tool_call.id,
            })

        elif tool_call.function.name == "create_ticket":
            domain = arguments.get("domain")
            status = arguments.get("status")
            specialist_notes = arguments.get("specialist_notes")

            response = create_ticket(domain, status, specialist_notes)
            responses.append({
                "role": "tool",
                "content": response,
                "tool_call_id": tool_call.id,
            })

        elif tool_call.function.name == "ask_specialist":
            domain = arguments.get("domain")
            specialist_message = arguments.get("message")

            specialist_history = ask_specialist(domain, specialist_message)
            response = json.dumps(specialist_history, ensure_ascii=False)
            responses.append({
                "role": "tool",
                "content": response,
                "tool_call_id": tool_call.id,
            })

    return responses


In [77]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)

        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    return (
        history,
        network_specialist_history,
        billing_specialist_history,
        security_specialist_history,
    )

In [78]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


def reset_histories():
    network_specialist_history.clear()
    billing_specialist_history.clear()
    security_specialist_history.clear()
    return [], [], [], []

In [79]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages", label="Customer Support Chatbot")

    with gr.Row():
        network_specialist_chatbot = gr.Chatbot(height=500, type="messages", label="Network Specialist Chatbot")
        billing_specialist_chatbot = gr.Chatbot(height=500, type="messages", label="Billing Specialist Chatbot")
        security_specialist_chatbot = gr.Chatbot(height=500, type="messages", label="Security Specialist Chatbot")

    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat,
        inputs=[chatbot],
        outputs=[
            chatbot,
            network_specialist_chatbot,
            billing_specialist_chatbot,
            security_specialist_chatbot,
        ],
    )

    ui.load(
        reset_histories,
        outputs=[
            chatbot,
            network_specialist_chatbot,
            billing_specialist_chatbot,
            security_specialist_chatbot,
        ],
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Specialist called for domain billing
